# Iris Flower Classification #

## 1. Understand the Dataset

##### Step 1 - Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, classification_report



##### Step 2 - Load Dataset

In [ ]:
iris =  load_iris()

df = pd.DataFrame(iris.data, columns=iris.feature_names)

df['species'] = iris.target

df.head()

##### Step 3 - Explore Dataset

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df['species'].value_counts()

##### Step 4 - Class Distribution

In [ ]:
df['species_name'] =  df['species'].map({
    0:"Setosa",
    1:'Versicolor',
    2:"Virginica"
})

In [ ]:
sns.countplot(data=df, x="species_name")

plt.title("Iris Species Distribution")
plt.xlabel("species")
plt.ylabel("Count")
plt.show()

##### Step 5 - Feature Distribution

In [ ]:
features = iris.feature_names

df[features].hist(
    figsize=(12,8),
    bins=20,
    edgecolor='black'
)

plt.suptitle("Feature Distributions")
plt.tight_layout()
plt.show()

##### Step 6 - Pairplot

In [ ]:
sns.pairplot(
    df,
    vars=features,
    hue='species_name',
    diag_kind="hist"
)

plt.show()

##### Step 7 - Correlation Heatmap

In [ ]:
corr = df[features].corr()

plt.figure(figsize=(8, 6))

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Feature Correlation Matrix")
plt.show()


##### Step 8 - Boxplots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, feature in zip(axes.flat, features):
    sns.boxplot(
        data=df,
        x="species_name",
        y=feature,
        ax=ax
    )
    ax.set_title(feature)

plt.tight_layout()
plt.show()


## 2. Data Preprocessing

##### Step 1 - Checking Data Quality

In [ ]:
df.isnull().sum()

In [ ]:
df.dtypes

##### Step 2 - Separating Features and Target

In [ ]:
x = df[iris.feature_names]
y = df['species']

##### Step 3 - Train/Test Split

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
    )


##### Step 4 - Feature Scaling

In [ ]:
scaler = StandardScaler()

x_train_scaled =  scaler.fit_transform(x_train)
x_test_scaled =  scaler.transform(x_test)

In [ ]:
scaled_df = pd.DataFrame(
    x_train_scaled,
    columns=x_train.columns
    )

scaled_df.head()


## 3. Model Training 

Why choose Logistic Regression for an Iris project?


Because it is suitable as a starting point for these reasons:
- Relatively easy to understand: You learn weights (Weights) for Features.
- Fast to train: especially since Iris data is small.
- Multiclass Classification supports: We have 3 types, not 2.
- Probabilities appear: We can know the probability of each type using predict_proba().
- Baseline Model is excellent: we can compare it to KNN and Decision Tree later.

### Create & Train

In [ ]:
model = LogisticRegression(max_iter=1000, solver="lbfgs")

In [ ]:
model.fit(x_train_scaled, y_train)

### Make Predictions

In [ ]:
y_pred  = model.predict(x_test_scaled)
y_pred

In [ ]:
y_test.to_numpy()

In [ ]:
results = pd.DataFrame({
    "Actual": y_test.to_numpy(),
    "Predicted": y_pred
})

results["Actual_Name"] = results["Actual"].map(
    dict(enumerate(iris.target_names))
)

results["Predicted_Name"] = results["Predicted"].map(
    dict(enumerate(iris.target_names))
)

results.head()


In [ ]:
probabilities = model.predict_proba(x_test_scaled)

probabilities[:5]

### Evaluation

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.2%}")

### Predict a New Flower

In [ ]:
new_flower = pd.DataFrame(
    [[5.0, 3.5, 1.5, 0.2]],
    columns=iris.feature_names
)

new_flower_scaled = scaler.transform(new_flower)

prediction = model.predict(new_flower_scaled)

flower_name = iris.target_names[prediction[0]]

flower_name


In [ ]:
probabilities = model.predict_proba(
    new_flower_scaled
)[0]

for flower, prob in zip(iris.target_names, probabilities):
    print(f"{flower}: {prob:.2%}")

## 4. Model Evaluation

In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm

In [ ]:
plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=iris.target_names,
    yticklabels=iris.target_names
)

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("Iris Confusion Matrix")

plt.show()


In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f'{accuracy:.2%}')

In [ ]:
precision = precision_score(y_test, y_pred, average=None)

In [ ]:
for name, score in zip(iris.target_names, precision):
    print(f"{name}: {score:.2%}")

In [ ]:
recall = recall_score(y_test, y_pred, average=None)

In [ ]:
for name, score in zip(iris.target_names, recall):
    print(f"{name}: {score:.2%}")

In [ ]:
f1 = f1_score(y_test, y_pred, average=None)

In [ ]:
for name, score in zip(iris.target_names, f1):
    print(f"{name}: {score:.2%}")

In [ ]:

print("Macro F1:", f1_score(
    y_test, y_pred, average="macro"
))

print("Weighted F1:", f1_score(
    y_test, y_pred, average="weighted"
))

print("Micro F1:", f1_score(
    y_test, y_pred, average="micro"
))


In [ ]:
report = classification_report(y_test, y_pred, 
                               target_names=iris.target_names,
                               digits=2)

In [ ]:
print(report)

In [ ]:
# Create a copy of test features
errors = x_test.copy()

# Add actual flower names
errors["Actual"] = y_test.map(
    dict(enumerate(iris.target_names))
)

# Add predicted flower names
errors["Predicted"] = pd.Series(
    y_pred,
    index=y_test.index
).map(dict(enumerate(iris.target_names)))

# Keep only wrong predictions
errors = errors[
    errors["Actual"] != errors["Predicted"]
]

errors


In [ ]:
wrong_mask = y_test.to_numpy() != y_pred

wrong_probabilities = model.predict_proba(
    x_test_scaled[wrong_mask]
)

print(wrong_probabilities.round(4))


## 5. Save Model

In [ ]:
import joblib

joblib.dump(model, "iris_model.joblib")
joblib.dump(scaler, "iris_scaler.joblib")

print("Model and Scaler saved successfully!")


In [ ]:

loaded_model = joblib.load("iris_model.joblib")

loaded_scaler = joblib.load("iris_scaler.joblib")

print("Model loaded successfully!")


In [ ]:

# Create new flower data
new_flower = pd.DataFrame(
    [[5.1, 3.5, 1.4, 0.2]],
    columns=iris.feature_names
)

# Scale the input
new_flower_scaled = loaded_scaler.transform(new_flower)

# Predict species
prediction = loaded_model.predict(new_flower_scaled)

# Display result
print("Predicted Species:", iris.target_names[prediction[0]])
